# 03. Custom CNN Trained from Scratch

Trains the engineered 4-stage convolutional neural network from scratch on CIFAR-100 (32x32) with data augmentation and cosine annealing learning rate schedule.

In [ ]:
import torch
import torch.nn as nn
from cv_benchmark.utils import set_seed, get_device, count_parameters
from cv_benchmark.data import get_dataloaders
from cv_benchmark.models import create_model
from cv_benchmark.training import Trainer
from cv_benchmark.evaluation import evaluate_model, profile_model_efficiency
from cv_benchmark.visualization import plot_training_curves

set_seed(42)
device = get_device("cuda")
print(f"Active Device: {device}")

loaders = get_dataloaders(model_name="custom_cnn", batch_size=128, image_size=(32, 32))
model, _ = create_model("custom_cnn", num_classes=100)
tot_p, tr_p = count_parameters(model)
print(f"Total Parameters: {tot_p:,} | Trainable: {tr_p:,}")

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=25)

trainer = Trainer(
    model=model,
    train_loader=loaders["train"],
    val_loader=loaders["val"],
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    experiment_name="custom_cnn_scratch",
)
history = trainer.fit(epochs=25)
plot_training_curves(history, save_path="reports/figures/custom_cnn_curves.png", title="Custom CNN Scratch")

metrics = evaluate_model(model, loaders["test"], device=device, num_classes=100)
print(f"Test Top-1 Accuracy: {metrics['top1_accuracy']}%")
print(f"Test Top-5 Accuracy: {metrics['top5_accuracy']}%")
print(f"Macro F1-Score: {metrics['f1_macro']}")